<a href="https://colab.research.google.com/github/jayasuriyajs3/Jayasuriya-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('imported successfully')
print(f'pandas : {pd.__version__}')
print(f'numpy : {np.__version__}')
print(f'requests : {requests.__version__}')

imported successfully
pandas : 2.2.2
numpy : 2.0.2
requests : 2.32.4


ETL
Level 1 - Simple Explanation
ETL stands for Extract, Transform, Load. It is a three-step process: first you EXTRACT data from a source (a file, a website, an API), then you
TRANSFORM it (clean it, fix errors, reshape it), then you LOAD it into a destination (a database, a dashboard, a file). This is how raw, messy data
becomes clean, usable data.

ETL - Extract Transform Load

Transform BEFORE loading

Data cleaned outside the database

Traditional approach (1970s-2000s)

Good for: relational databases with fixed schemas

Used when: storage is expensive, data must be clean before
entry

Tools: Python (Pandas), Apache Spark, Talend

ELT - Extract Load Transform

Load raw data first, transform inside the warehouse

Modern approach (2010s-present)

Good for: cloud data warehouses (Snowflake, BigQuery)

Used when: storage is cheap, transformation power is scalable

Preserves raw data - can re-transform any time

Tools: dbt, Fivetran, Airbyte, SQL-based transforms


In [2]:
df=pd.read_csv('/content/messy_sales_data.csv')
print('imported')
print("columns:",df.shape[1])
print("rows:",df.shape[0])
print(df.columns.tolist())


imported
columns: 9
rows: 30
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


In [3]:
print('='*60)
print('DATA QUALITY DIAGNOSIS REPORT')
print('='*60)
print("No of missing vales in each column :")
print(df.isnull().sum())
print('-'*60)
print("Duplicate rows : ",df.duplicated().sum())
print('-'*60)
print("Data types of each column :")
print(df.dtypes)
print('-'*60)
print("Unique categories :", df['category'].unique())
print('-'*60)
print("Sample customer name:",df['customer_name'].dropna().unique()[:8])
print('-'*60)
print("Sample order date vales: ", df['order_date'].unique()[:8])

DATA QUALITY DIAGNOSIS REPORT
No of missing vales in each column :
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
------------------------------------------------------------
Duplicate rows :  0
------------------------------------------------------------
Data types of each column :
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
------------------------------------------------------------
Unique categories : ['Electronics' 'Accessories' nan]
------------------------------------------------------------
Sample customer name: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
---------------------------

In [4]:
c_df=df.copy()
print("working copy created",c_df.shape)

print("Total null values :", c_df.isnull().sum().sum())

c_df['customer_name'].fillna('Unknown Customer',inplace=True)

median_qty=df['quantity'].median()
c_df['quantity'].fillna(median_qty,inplace=True)
print(f'Filled missing quantity with median :{median_qty}')

c_df['category'].fillna('Uncategorized',inplace=True)

print('After Fixing nulls:',c_df.isnull().sum().sum())

working copy created (30, 9)
Total null values : 7
Filled missing quantity with median :2.0
After Fixing nulls: 1


In [5]:
print(f'Before Duplication: {len(df)} rows')
print(f'Duplicated rows: {c_df.duplicated().sum()}')
print('\nDuplicate rows :')
print(c_df[c_df.duplicated(keep=False)][['order_id','customer_name','product','order_date']].head())
c_df.drop_duplicates(inplace=True)
print(f'\nAfter Duplication: {len(c_df)} rows')
print(f'Rows removed :{len(df)-len(c_df)}')


Before Duplication: 30 rows
Duplicated rows: 0

Duplicate rows :
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After Duplication: 30 rows
Rows removed :0


In [6]:
print('Sample dates before parsing')
print(c_df['order_date'].head(10).tolist())

c_df['order_date']=pd.to_datetime(c_df['order_date'],dayfirst=False,errors='coerce' )

print('Sample dates after parsing')
print(c_df['order_date'].head(10).tolist())

c_df['year'] = c_df['order_date'].dt.year
c_df['month']= c_df['order_date'].dt.month
c_df['month_name']= c_df['order_date'].dt.strftime('%B')
print('\nSample dates after parsing')
print(c_df[['order_date','year','month','month_name']].head(10))

Sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']
Sample dates after parsing
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00'), Timestamp('2024-01-15 00:00:00'), Timestamp('2024-01-15 00:00:00')]

Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN
6 2024-01-12  2024.0    1.0    January
7 2024-01-13  2024.0    1.0    January
8 2024-01-15  2024.0    1.0    January
9 2024-01-15  2024.0    1.0    January


In [7]:
print('Before Standardization :',c_df['customer_name'].unique()[:6])
c_df['customer_name']=(c_df['customer_name'].str.strip().str.title())
print('After Standardization :',c_df['customer_name'].unique()[:6])

print(f'before : Keyboard rows with electronics category:')
wrong_mask = (c_df['product']=='keyboard') & (c_df['category']=='Electronics')
print(c_df[wrong_mask][['product','category']])
c_df.loc[wrong_mask, 'category'] = 'Accessories'
print('After fix:Unique categories', c_df['category'].unique())

Before Standardization : ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization : ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']
before : Keyboard rows with electronics category:
Empty DataFrame
Columns: [product, category]
Index: []
After fix:Unique categories ['Electronics' 'Accessories' 'Uncategorized']


In [8]:
c_df['quantity']=pd.to_numeric(c_df['quantity'],errors='coerce')
c_df['unit_price']=pd.to_numeric(c_df['unit_price'],errors='coerce')
c_df['revenue']=c_df['quantity']*c_df['unit_price']
print('Revenue column created:')
print(c_df[['customer_name','product','quantity','unit_price','revenue']].head())
print(f"\n Total Revenue across all orders: rs{c_df['revenue'].sum():,.0f}")


Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    Amit Verma  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

 Total Revenue across all orders: rs818,000


In [9]:
print('=' * 55)
print('   POST-CLEANING VALIDATION REPORT')
print('=' * 55)
print(f"Orginaal Rows : {len(c_df)}(duplicates)")
print(f"Cleaned Rows : {len(c_df)}")
print(f"Rows Removed : {len(df) - len(c_df)}(duplicates)")
print(f"Missing Values : {c_df.isnull().sum().sum()}")
print(f"Duplicates : {c_df.duplicated().sum()}")
print(f"Date Nulls : {c_df['order_date'].isnull().sum()}")
print(f"Revenue Nan : {c_df['revenue'].isnull().sum()}")
print(f"Categories : {sorted(c_df["category"].unique())}")
print('=' *55)

all_clean = (
    df.isnull().sum().sum()==0 and
    df.duplicated().sum()==0
)
print(f'Data is clean :{all_clean}')

   POST-CLEANING VALIDATION REPORT
Orginaal Rows : 30(duplicates)
Cleaned Rows : 30
Rows Removed : 0(duplicates)
Missing Values : 9
Duplicates : 0
Date Nulls : 2
Revenue Nan : 0
Categories : ['Accessories', 'Electronics', 'Uncategorized']
Data is clean :False


In [10]:
API_KEY='4ab53fa98641345b8576f32d1be8c651'
BASE_URL='https://api.openweathermap.org/data/2.5/weather'
CITIES=['Mumbai','Delhi','Bangalore','Hyderabad','Ahmedabad']
print(f'API Configured for {len(CITIES)} cities')
print(f'cities: {CITIES}')

API Configured for 5 cities
cities: ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Ahmedabad']


In [16]:
import requests
import pandas as pd
from datetime import datetime

API_KEY = '4ab53fa98641345b8576f32d1be8c651'
BASE_URL = 'https://api.openweathermap.org/data/2.5/weather'


cities = ["Chennai", "Coimbatore", "Delhi", "Mumbai"]

weather_list = []


for city in cities:

    params = {
        'q': city,
        'appid': API_KEY,
        'units': 'metric'
    }

    response = requests.get(BASE_URL, params=params)
    data = response.json()

    if response.status_code == 200:

        weather_data = {
            'City': data['name'],
            'Country': data['sys']['country'],
            'Temperature (°C)': data['main']['temp'],
            'Humidity (%)': data['main']['humidity'],
            'Weather': data['weather'][0]['main'],
            'Description': data['weather'][0]['description'],
            'Wind Speed (m/s)': data['wind']['speed'],
            'Date & Time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

        weather_list.append(weather_data)

    else:
        print(f"Could not fetch data for {city}")


df = pd.DataFrame(weather_list)


df.to_csv("weather_data.csv", index=False)

print(df)
print("\nData saved successfully!")

         City Country  Temperature (°C)  Humidity (%) Weather  \
0     Chennai      IN             32.80            71  Clouds   
1  Coimbatore      IN             28.88            65    Haze   
2       Delhi      IN             36.05            44    Rain   
3      Mumbai      IN             31.99            66    Haze   

        Description  Wind Speed (m/s)          Date & Time  
0  scattered clouds              5.14  2026-05-28 13:17:31  
1              haze              7.20  2026-05-28 13:17:31  
2        light rain              7.20  2026-05-28 13:17:31  
3              haze              4.63  2026-05-28 13:17:31  

Data saved successfully!


In [17]:
import pandas as pd

data = {
    "Employee_ID": [101,102,103,104,105,106,107,108,102,105,109,110],
    "Name": ["Arun","Priya","Kavin","Meena","Rahul","Divya","Suresh","Anitha","Priya","Rahul","Vignesh","Keerthana"],
    "Department": ["HR","IT","Sales","Finance","Marketing","IT","HR","Sales","IT","Marketing","Finance","IT"],
    "Monthly_Salary": [25000,40000,30000,45000,35000,42000,28000,32000,40000,35000,50000,47000]
}

df = pd.DataFrame(data)

# Save as Excel
df.to_excel("employee_data.xlsx", index=False)

print("Excel file created successfully!")

Excel file created successfully!


In [18]:
import pandas as pd

df = pd.read_excel("employee_data.xlsx")

print("Original Data:")
print(df)

df = df.drop_duplicates()

df["Yearly_Salary"] = df["Monthly_Salary"] * 12

df.columns = df.columns.str.strip()

df.to_csv("cleaned_employee_data.csv", index=False)

df.to_excel("cleaned_employee_data.xlsx", index=False)

print("\nCleaned Data:")
print(df)

print("\nETL Process Completed Successfully!")

Original Data:
    Employee_ID       Name Department  Monthly_Salary
0           101       Arun         HR           25000
1           102      Priya         IT           40000
2           103      Kavin      Sales           30000
3           104      Meena    Finance           45000
4           105      Rahul  Marketing           35000
5           106      Divya         IT           42000
6           107     Suresh         HR           28000
7           108     Anitha      Sales           32000
8           102      Priya         IT           40000
9           105      Rahul  Marketing           35000
10          109    Vignesh    Finance           50000
11          110  Keerthana         IT           47000

Cleaned Data:
    Employee_ID       Name Department  Monthly_Salary  Yearly_Salary
0           101       Arun         HR           25000         300000
1           102      Priya         IT           40000         480000
2           103      Kavin      Sales           30000        